# CellSize-Clust — walkthrough notebook

Step-by-step exploration of the analytical pipeline. For a one-command end-to-end run, use `scripts/run_analysis.py` instead.

## Sections
1. Load and inspect the dataset
2. Per-muscle z-scoring
3. Fit the 8 algorithms at K = 2
4. Internal validation indices and composite score
5. ARI agreement between algorithms
6. Cluster signatures (Low vs High)
7. Animal-level inference: NCD vs HFD


In [ ]:
import sys
from pathlib import Path

# Make src/ importable
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from cellclust import config
from cellclust.data import load_data, split_by_muscle, summarize_strata
from cellclust.clustering import fit_predict, reorder_clusters_by_mean
from cellclust.validation import internal_indices, composite_score, ari_matrix
from cellclust.inferential import per_animal_proportions, diet_effect_summary, chi2_stratified_by_diet
from cellclust.plots import configure_rcparams, plot_cluster_signatures

configure_rcparams()
print('Version:', __import__('cellclust').__version__)
print('Algorithms:', config.ALGORITHMS)
print('K_MAIN:', config.K_MAIN)

## 1. Load and inspect the dataset

In [ ]:
df = load_data(config.RAW_CSV)
print(f'Total fibers: {len(df):,}')
print(f'Animals     : {df["animal"].nunique()}')
summarize_strata(df)

## 2. Per-muscle z-scoring

In [ ]:
data_by_muscle = split_by_muscle(df)
X_by_muscle = {
    m: StandardScaler().fit_transform(
        data_by_muscle[m]['feret'].values.reshape(-1, 1)
    )
    for m in config.MUSCLES
}
for m, X in X_by_muscle.items():
    print(f'{m}: shape={X.shape}  mean={X.mean():.4f}  sd={X.std():.4f}')

## 3. Fit the 8 algorithms at K = 2

In [ ]:
results = []
labels_by_alg = {m: {} for m in config.MUSCLES}
for m in config.MUSCLES:
    X = X_by_muscle[m]
    for alg in config.ALGORITHMS:
        labels = fit_predict(alg, X, k=2)
        labels_by_alg[m][alg] = labels
        idx = internal_indices(X, labels)
        results.append({'Muscle': m, 'Algorithm': alg, 'k': 2, **idx})

bench = composite_score(pd.DataFrame(results))
bench.sort_values(['Muscle', 'Composite'], ascending=[True, False])

## 4. ARI matrices

In [ ]:
for m in config.MUSCLES:
    mat = ari_matrix(labels_by_alg[m])
    print(f'\n=== {m} — ARI matrix ===')
    print(mat.round(2))

## 5. Cluster signatures (Agglom-Ward, K = 2)

In [ ]:
principal = config.PRINCIPAL_ALGORITHM
labels_principal = {
    m: reorder_clusters_by_mean(
        labels_by_alg[m][principal],
        data_by_muscle[m]['feret'].values,
    )
    for m in config.MUSCLES
}
fig = plot_cluster_signatures(data_by_muscle, labels_principal,
                                cluster_names=('Low', 'High'))
plt.show()

## 6. Animal-level inference

In [ ]:
CLUSTER_NAMES = ['Low', 'High']
frames = []
for m in config.MUSCLES:
    sub = data_by_muscle[m].copy()
    sub['cluster'] = pd.Categorical(
        [CLUSTER_NAMES[i] for i in labels_principal[m]],
        categories=CLUSTER_NAMES,
    )
    frames.append(sub)
assignments = pd.concat(frames, ignore_index=True)

per_animal = per_animal_proportions(assignments, cluster_names=CLUSTER_NAMES)
print(f'Per-animal observations: {len(per_animal)}')
per_animal.head()

In [ ]:
diet_eff = diet_effect_summary(
    per_animal,
    muscles=config.MUSCLES,
    cluster_names=CLUSTER_NAMES,
    n_boot=500,
)
diet_eff[['Muscle', 'Cluster', 'NCD_mean', 'HFD_mean',
          'r_rb', 'CI_lo', 'CI_hi', 'Cohen_d']]

In [ ]:
chi2_stratified_by_diet(assignments, cluster_names=CLUSTER_NAMES)